In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, transform, struct
from pyspark.sql.window import Window
import os
import sys
from pyspark.sql.functions import explode
sys.path.append(os.path.abspath(".."))
from common.schema import MovieSchema

1. SparkSession chính là entry point để chương trình Python giao tiếp với hệ thống Apache Spark. 
2. Nếu chưa tạo SparkSession thì chương trình Python không có kết nối nào với Spark cluster hoặc Spark engine, nên bạn không thể xử lý dữ liệu bằng Spark.
3. SparkSession \
Là entry point của Spark API \
Tạo kết nối giữa Python và Spark engine (JVM) \
Khởi tạo SparkContext và cấu hình Spark \
Cho phép sử dụng DataFrame, SQL, RDD, streaming…

In [2]:
host = "spark://localhost:7077"
# spark = SparkSession.builder.appName("Spark Transformation").master(host).getOrCreate()
spark = SparkSession.builder.appName("TMDB Transformation").master("local[*]").getOrCreate()

In [3]:
spark

1. Cột raw_payload là kiểu struct chứ không phải cột phẳng. 
2. Vì vậy không thể dùng df["raw_payload"] để lấy trực tiếp các trường bên trong.

In [4]:
base_path = os.path.abspath(os.path.join(os.getcwd(), "../.."))
file_path = os.path.join(
    base_path,
    "dataset",
    "tmdb",
    "movies",
    "trending_114240.jsonl"
)
df = spark.read.json(file_path)
# Đọc dữ liệu và bóc tách lớp raw_payload ngay lập tức
df.printSchema()

root
 |-- metadata: struct (nullable = true)
 |    |-- entity: string (nullable = true)
 |    |-- extraction_method: string (nullable = true)
 |    |-- http_status: long (nullable = true)
 |    |-- ingestion_id: string (nullable = true)
 |    |-- ingestion_timestamp: string (nullable = true)
 |    |-- raw_hash: string (nullable = true)
 |    |-- search_parameters: struct (nullable = true)
 |    |    |-- movie_id: long (nullable = true)
 |    |-- source_system: string (nullable = true)
 |-- raw_payload: struct (nullable = true)
 |    |-- adult: boolean (nullable = true)
 |    |-- backdrop_path: string (nullable = true)
 |    |-- belongs_to_collection: string (nullable = true)
 |    |-- budget: long (nullable = true)
 |    |-- genres: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- homepage: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- imdb_id: string (nullable = true)
 |    |-- origin_country: array (nullable = true)
 |  

In [ ]:
metadata = df.select("metadata.*")
transform_metadata = metadata.select(
    col("ingestion_id"),
    col("ingestion_timestamp"),
    col("raw_hash")
)
transform_metadata.show(truncate=False)
raw_payload = df.select("raw_payload.*")
raw_payload.show(truncate=False)

+------------------------------------+--------------------------+----------------------------------------------------------------+
|ingestion_id                        |ingestion_timestamp       |raw_hash                                                        |
+------------------------------------+--------------------------+----------------------------------------------------------------+
|966801eb-f2e3-43db-a4a5-a1bc2e477140|2026-03-13T11:42:40.965557|074e201e193be54578e33131b1df8a55e94e276c0fe81593b32d81e9fce5aab3|
+------------------------------------+--------------------------+----------------------------------------------------------------+

+-----+--------------------------------+---------------------+------+-----------------------------------+--------------------------------------+-------+----------+--------------+-----------------+--------------+-------------------------------------------------------------------------------------------------------------------------------------

1. Bóc toàn bộ dữ liệu các field dạng JSON bên trong một cột dạng struct và đưa chúng ra thành các cột riêng của DataFrame.
- Ví dụ: \
root \
 |-- raw_payload: struct \
 |.........|-- imdb_id: string \
 |.........|-- title: string \
 |.........|-- genres: array<string> \
 |.........|-- runtime: long \
 |.........|-- release_date: string 

 - Sẽ trở thành \
 root \
 |-- imdb_id: string \
 |-- title: string \
 |-- genres: array<string> \
 |-- runtime: long \
 |-- release_date: string 

Hàm	        Kết quả \
first()	    lấy dòng đầu tiên và trả về 1 Row\
head()	    giống first() \
take(n)	    lấy n dòng \
collect()	lấy toàn bộ dữ liệu về driver và trả về 1 list các Row

1. Spark DataFrame là immutable (không thay đổi trực tiếp).
2. Hàm withColumnsRenamed() trả về một DataFrame mới, nhưng bạn không gán lại cho biến data

- Bước tiếp theo: Chuẩn hóa dữ liệu của json trả về theo 1 schma đã thiết kế từ trước

In [14]:
data = df.select(
    col("metadata.ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
    col("raw_payload.*")
)
transformed_df = data.select(
    col("ingestion_timestamp"),
    col("imdb_id"),
    col("title").alias("movie_title"),
    col("genres"),
    to_date(col("release_date")).alias("release_date"),
    col("overview").alias("description"),
    (col("runtime") * 60).cast("long").alias("duration_seconds"),
    col("vote_average").alias("rating"),
    col("vote_count"),
    col("original_language"),
    transform("production_companies", 
              lambda x: struct(x["name"], x["origin_country"])).alias("production_companies")
)
silver_df = transformed_df
silver_df.show(truncate = False)

+--------------------------+----------+-----------+-----------------------------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+------+----------+-----------------+-------------------------------------------------------------------------------------+
|ingestion_timestamp       |imdb_id   |movie_title|genres                             |release_date|description                                                                                                                                            |duration_seconds|rating|vote_count|original_language|production_companies                                                                 |
+--------------------------+----------+-----------+-----------------------------------+------------+--------------------------------------------------------------------------------------------------------------------

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
#Lọc ra các bản ghi thiếu imdb_id
silver_df = silver_df.filter(col("imdb_id").isNotNull())
""" Deduplicate data
    Sử dụng window function để loại bỏ trùng
    Ở đây chúng ta sẽ phân nhóm theo imdb_id và sắp xếp các record có cùng imdb_id 
    theo thời gian cào mới nhất 
    Tạo thêm row_number để đánh số thứ tự từng dòng trong mỗi partition.
"""
window_spec = Window.partitionBy(col("imdb_id")).orderBy(col("ingestion_timestamp").desc())
rn = row_number().over(window_spec)

"""
    Giải thích câu lệnh phía dưới:
    Thêm cột rn ở trên vào dataFrame sau đó thì lọc ra cột rn có rn = 1
    Rồi drop 2 cột rn và ingestion_timestamp cho đúng với schema

"""
dedup_silver_df = silver_df.withColumn("rn", rn) \
        .filter(col("rn") == 1) \
        .drop("rn", "ingestion_timestamp")

dedup_silver_df.show(truncate=False)

+----------+-----------+-----------------------------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+------+----------+-----------------+-------------------------------------------------------------------------------------+
|imdb_id   |movie_title|genres                             |release_date|description                                                                                                                                            |duration_seconds|rating|vote_count|original_language|production_companies                                                                 |
+----------+-----------+-----------------------------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+------+----------+----------